# Cycle 4 — PokéAPI: Berry Flavors and Contest Types

**Goal:** get data from two PokéAPI pages, check it, and put it in a table (DataFrame).

## Step 1: Read the documentation
- **Pages (endpoints):** `https://pokeapi.co/api/v2/berry-flavor/spicy/` and `https://pokeapi.co/api/v2/contest-type/cool/`
- **Request type:** GET. **Parameters:** none. **Login:** none.
- **Rules:** it is free. Save the data so we don't ask again and again. The data comes from PokéAPI.
- **Fixtures:** saved copies of the example data from the docs. They let the notebook work with no internet and give the same result every time.

## Step 2: Set up and load the saved fixtures
This cell sets the folder paths and checks the folder is there. If it isn't, we get a clear message.

In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd

DATA_DIR = Path("datasets")
FIXTURE_DIR = DATA_DIR / "api-fixtures"
assert FIXTURE_DIR.exists(), "Run from the notebook's folder and keep datasets next to it."

In [ ]:
flavor_path = FIXTURE_DIR / "pokeapi-berry-flavor-spicy.json"
contest_path = FIXTURE_DIR / "pokeapi-contest-type-cool.json"

with flavor_path.open(encoding="utf-8") as handle:
    flavor_fixture = json.load(handle)
with contest_path.open(encoding="utf-8") as handle:
    contest_fixture = json.load(handle)

print(flavor_fixture.keys())
print(contest_fixture.keys())

## Step 3: Endpoints, parameters, and expected fields
These are the web addresses, an empty parameters dictionary (we need none), and the fields each reply must have.

In [ ]:
FLAVOR_ENDPOINT = "https://pokeapi.co/api/v2/berry-flavor/spicy/"
CONTEST_ENDPOINT = "https://pokeapi.co/api/v2/contest-type/cool/"
PARAMS = {}
EXPECTED_FLAVOR = {"id", "name", "berries", "contest_type"}
EXPECTED_CONTEST = {"id", "name", "berry_flavor", "names"}

## Step 4: Check the data and get it
First, a function that checks the reply has the fields we need. A reply can look fine and still be missing data.

Then a function that gets the data. It does this in order:
1. Starts with the saved fixture.
2. If `use_live` is on, it asks the website for the data. It waits at most **10 seconds**.
3. Checks the **status code**. This catches errors like 404.
4. Reads the **JSON**.
5. Checks the **fields**.
6. If anything goes wrong, it prints a message and uses the fixture.

`USE_LIVE` is off by default, so the output should say `cached fixture` twice.

In [ ]:
def validate_payload(payload, expected):
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")
    missing = expected - set(payload)
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")
    return payload

validate_payload(flavor_fixture, EXPECTED_FLAVOR)
validate_payload(contest_fixture, EXPECTED_CONTEST)

In [ ]:
def acquire(endpoint, fixture, expected, use_live=False):
    source = "cached fixture"
    payload = fixture
    if use_live:
        try:
            response = requests.get(endpoint, params=PARAMS, timeout=10)
            response.raise_for_status()
            payload = response.json()
            validate_payload(payload, expected)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = fixture
    return validate_payload(payload, expected), source

USE_LIVE = False  # set to True (with internet) to use live data

flavor, flavor_source = acquire(FLAVOR_ENDPOINT, flavor_fixture, EXPECTED_FLAVOR, use_live=USE_LIVE)
contest, contest_source = acquire(CONTEST_ENDPOINT, contest_fixture, EXPECTED_CONTEST, use_live=USE_LIVE)
print(flavor_source, "|", contest_source)

## Step 5: Put the data in tables
The `berries` part of the JSON is a list. I loop through it and add one row for each berry (`flavor`, `berry`, `potency`). The contest type gets its own small table. Then I join (merge) the two tables using `flavor`, so each berry row also shows its contest type.

In [ ]:
rows = {"flavor": [], "berry": [], "potency": []}
for item in flavor["berries"]:
    rows["flavor"].append(flavor["name"])
    rows["berry"].append(item["berry"]["name"])
    rows["potency"].append(item["potency"])
flavor_df = pd.DataFrame(rows)

contest_df = pd.DataFrame({
    "flavor": [contest["berry_flavor"]["name"]],
    "contest_type": [contest["name"]],
})

assert flavor["contest_type"]["name"] == contest["name"], "Flavor and contest type do not match"

berries_df = flavor_df.merge(contest_df, on="flavor")
berries_df

## Step 6: Check the table
Before using the data, I check its size, data types, missing values, repeats, and whether the numbers make sense.

In [ ]:
print("Shape (rows, columns):", berries_df.shape)
print()
print("Data types:")
print(berries_df.dtypes)
print()
print("Missing values per column:")
print(berries_df.isna().sum())
print()
print("Repeated rows:", berries_df.duplicated().sum())
print("Repeated berries:", berries_df["berry"].duplicated().sum())
print("Potency range:", berries_df["potency"].min(), "to", berries_df["potency"].max())

**What the checks show (using the fixture):**
- The table has **1 row and 4 columns**. It is small because the fixture only has one berry (`rowap`).
- `flavor`, `berry`, and `contest_type` are text. `potency` is a whole number. That is right, because I need potency to be a number to do math with it.
- There are **no missing values and no repeats**, so I didn't need to fix anything.
- Potency is **10**. That is a normal positive number.

**Why this matters later:** with only 1 row, the average is just that one number. If I set `USE_LIVE = True`, I get all the spicy berries. These checks would then catch repeats or negative numbers before they mess up the average.

In [ ]:
assert berries_df.isna().sum().sum() == 0, "There are missing values"
assert berries_df["potency"].min() >= 0, "Potency should not be negative"

The checks passed.

## Step 7: Write down where the data came from
This notes the source and rules, so someone else can check my work or do it again.

In [ ]:
provenance = {
    "provider": "PokéAPI (pokeapi.co)",
    "endpoints": [FLAVOR_ENDPOINT, CONTEST_ENDPOINT],
    "parameters": PARAMS,
    "acquisition_source": {"berry-flavor": flavor_source, "contest-type": contest_source},
    "fixture_note": "Fixtures are the docs' example responses",
    "license": "Data from pokeapi.co; attribution required; classroom use",
}
provenance

## Step 8: One observation

In [ ]:
print("Number of berries:", len(berries_df))
print("Average potency:", berries_df["potency"].mean())
print("Contest type for this flavor:", contest["name"])

**My observation:** The spicy flavor goes with the **cool** contest type. Its one berry in the fixture (rowap) has a potency of 10. *(Rewrite this in your own words. Change it if you use `USE_LIVE = True`.)*

## Reflection
My answers are in README.md.